# NestedSimPy — Dual Sourcing with Lookahead Expediting

This notebook runs a NestedSimPy-specific example: the dual-sourcing inventory model of Song, Xiao, Zhang and Zipkin (2017), where lead times are endogenous and in each decision epoch the decision-maker decides whether to place a normal or an expedited order. It uses `env.decide` and `set_inner_actions`.

See the [example page](https://nestedsimpy.github.io/official-parity/dual-sourcing.html) for the side-by-side plain/nested code.

## 1. Install

_Pre-release: NestedSimPy installs from a hosted wheel. (After the public release this becomes `pip install nestedsimpy`.)_

In [ ]:
# Pre-release install from a hosted wheel (Google Drive).
!pip install -q gdown
import gdown
gdown.download(id="1N7mlgDVpVids6Ekr4p2e-gEUrUodiEuq",
               output="nestedsimpy-0.1.0-py3-none-any.whl", quiet=True)
!pip install -q "nestedsimpy-0.1.0-py3-none-any.whl[plot]"

import nestedsimpy
print("NestedSimPy ready —", len(nestedsimpy.__all__), "public objects")

## 2. Run the nested example

The model is written to a file and run as a subprocess; the output below is the **outer** trajectory. In rollout mode it executes the lookahead picks, so it can differ from the plain example.

In [ ]:
%%writefile dual_sourcing_colab.py
from __future__ import annotations

# --- inline prelude (replaces the examples' local _imports shim) ---
import argparse, os, random, shutil, sys, itertools
from pathlib import Path

import simpy
import nestedsimpy
from nestedsimpy import (
    NestedEnvironment, NestedResource, NestedPreemptiveResource,
    NestedStore, NestedContainer,
)
try:
    from nestedsimpy.postprocess import (
        package_latest_run, relocate_raw_artifacts, export_realizations,
    )
except Exception:  # pragma: no cover
    package_latest_run = relocate_raw_artifacts = export_realizations = None

DEFAULT_OUT_ROOT = Path("nested_output")
DEFAULT_AUTOPLOT = False
REPO_ROOT = Path(".")
PACKAGE_ROOT = Path(".")

def default_out(*parts):
    p = DEFAULT_OUT_ROOT.joinpath(*map(str, parts)); p.mkdir(parents=True, exist_ok=True); return p

def set_nested_output_folder(*parts):
    p = Path(os.path.join(*[str(x) for x in parts])); p.mkdir(parents=True, exist_ok=True); return p
# --- end prelude ---

"""
SimPy simulation of the dual-sourcing inventory model of

    Song, J.-S., Xiao, L., Zhang, H., & Zipkin, P. (2017).
    "Optimal Policies for a Dual-Sourcing Inventory Problem with
    Endogenous Stochastic Leadtimes." Operations Research 65(2):379-395.

Model (paper, Section 3)
------------------------
* Single product, unit-sized Poisson demand (rate lambda), full
  backlogging, continuous review, no fixed order cost.
* The normal source is a two-stage tandem queue: a normal order joins
  server 1 (exp. rate mu1), then server 2 (exp. rate mu2), then reaches
  stock.  An emergency order skips server 1 and joins server 2 directly.
  Both servers are FIFO and work-conserving.  Lead times are therefore
  ENDOGENOUS: ordering more congests the queues and lengthens lead times.
* Costs -- all six components of the paper:
    c1  per unit ordered from the normal source (charged at placement)
    c2  per unit ordered from the emergency source, c2 > c1
    h1  holding per unit per unit time at server 1
    h2  holding per unit per unit time at server 2
    h   holding per unit per unit time on finished goods on hand
    b   backorder per unit per unit time
  The paper normalizes h1 to 0 by folding it into c1 (Section 3); the
  default below follows that convention, but h1 is a parameter, so a
  nonzero value is also handled.

Default parameter values are the instance of Table 5 (Section 6) with
h=1, b=60, h2=2: lambda=6, mu1=8, mu2=7, c1=10, c2=30.  For this
instance the paper reports (long-run average cost per unit time):
    optimal policy 96.5149 | TC policy 96.5349 | best DI policy 99.8904,
where the best Dual-Index (DI) policy uses s1=30, s2=12.  The
DualIndexPolicy below with those parameters is exactly the paper's DI
benchmark, so the simulation can be validated against 99.8904.

Policies
--------
A policy is a callable ``policy(state) -> (normal, emergency)`` giving
the number of units to order from each source now; it is consulted after
every demand and every state change of the supply system.  The paper's
DI policy (Section 6.2) maintains IP1 = IN + N2 + N1 at s1 and
IP2 = IN + N2 at or above s2.

This is the rollout version: the same policy object goes into
env.decide, which tries each candidate order in inner simulations at
every review and executes the best-scoring one.
"""



from dataclasses import dataclass

import numpy as np

# One entry per candidate decision, in the shape the policy returns:
# (normal, emergency) -- order nothing, one normal unit, one emergency
# unit, or both.  The baseline policy's own decision competes
# automatically -- it can be a larger order than any of these, e.g.
# the opening order that fills the position up to s1.
ACTIONS = [(0, 0), (1, 0), (0, 1), (1, 1)]
INNER_HORIZON = 3.0      # lookahead window, in time units
INNER_REPS = 12          # replications per action

NESTED_OUTPUT_FOLDER = set_nested_output_folder("simpy_examples",
                                                "dual_sourcing")


# ---------------------------------------------------------------------------
# Parameters (problem data only -- no logic).  Values: paper's Table 5,
# row h=1, b=60, h2=2.
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class Params:
    demand_rate: float = 6.0       # lambda: Poisson demand rate
    stage1_rate: float = 8.0       # mu1: exponential rate of server 1
    stage2_rate: float = 7.0       # mu2: exponential rate of server 2
    normal_cost: float = 10.0      # c1: $ per unit, normal source
    emergency_cost: float = 30.0   # c2: $ per unit, emergency source
    stage1_holding: float = 0.0    # h1: $/unit/time at server 1 (paper
    #                                normalizes h1 = 0, folding it into c1)
    stage2_holding: float = 2.0    # h2: $/unit/time at server 2
    holding_cost: float = 1.0      # h:  $/unit/time on hand
    backorder_cost: float = 60.0   # b:  $/unit/time backlogged
    horizon: float = 2000.0        # length of one simulation run
    warmup: float = 200.0          # costs before this time are discarded
    initial_net: int = 12          # on-hand stock at time 0, empty pipeline


# ---------------------------------------------------------------------------
# Observable system state and policies
# ---------------------------------------------------------------------------

@dataclass
class State:
    net: int      # IN: on-hand minus backlog (negative = backlogged)
    stage1: int   # N1: units waiting at or being processed by server 1
    stage2: int   # N2: units waiting at or being processed by server 2

    @property
    def ip1(self) -> int:
        """IP1 = IN + N2 + N1: position including both sources (eq. 24)."""
        return self.net + self.stage1 + self.stage2

    @property
    def ip2(self) -> int:
        """IP2 = IN + N2: position past server 1 (eq. 25)."""
        return self.net + self.stage2


class DualIndexPolicy:
    """The paper's Dual-Index (DI) policy, Section 6.2 (Song-Zipkin 2009).

    Maintain IP2 >= s2 with emergency orders, then IP1 = s1 with normal
    orders.  Set s2 = None to never use the emergency source.
    """

    def __init__(self, s1: int, s2: int | None = None):
        self.s1, self.s2 = s1, s2

    def __call__(self, state: State) -> tuple[int, int]:
        emergency = 0 if self.s2 is None else max(0, self.s2 - state.ip2)
        normal = max(0, self.s1 - state.ip1 - emergency)
        return normal, emergency

    def __repr__(self):
        return f"DualIndex(s1={self.s1}, s2={self.s2})"


# ---------------------------------------------------------------------------
# One replication, with the emergency decision handed to NestedSimPy
# ---------------------------------------------------------------------------

def simulate(params: Params, policy, seed: int, *,
             inner_horizon=INNER_HORIZON, inner_reps=INNER_REPS,
             out_dir=None) -> dict:
    """One rollout run; returns costs, counts and the environment."""
    p = params
    np.random.seed(seed)
    env = NestedEnvironment()
    server1 = NestedResource(env, capacity=1, nested_id="server1")
    server2 = NestedResource(env, capacity=1, nested_id="server2")
    state = State(net=p.initial_net, stage1=0, stage2=0)
    costs = {"holding": 0.0, "backorder": 0.0, "pipeline": 0.0,
             "ordering": 0.0}
    counts = {"normal": 0, "emergency": 0}
    last_accrual = [0.0]

    def accrue():
        """Charge all time-proportional costs since the last state change.

        Must be called BEFORE any change to net, stage1, or stage2.
        """
        dt = env.now - last_accrual[0]
        increment = (p.holding_cost * max(state.net, 0)
                     + p.backorder_cost * max(-state.net, 0)
                     + p.stage1_holding * state.stage1
                     + p.stage2_holding * state.stage2) * dt
        costs["holding"] += p.holding_cost * max(state.net, 0) * dt
        costs["backorder"] += p.backorder_cost * max(-state.net, 0) * dt
        costs["pipeline"] += (p.stage1_holding * state.stage1
                              + p.stage2_holding * state.stage2) * dt
        env.record("cost", increment)               # expose to the branches
        last_accrual[0] = env.now

    def produced_unit(emergency: bool):
        """Lifecycle of one ordered unit until it reaches inventory."""
        if not emergency:
            with server1.request() as turn:
                yield turn
                yield env.nested_timeout(
                    {"distribution": "exponential", "rate": p.stage1_rate})
            accrue()
            state.stage1 -= 1
            state.stage2 += 1
        with server2.request() as turn:
            yield turn
            yield env.nested_timeout(
                {"distribution": "exponential", "rate": p.stage2_rate})
        accrue()
        state.stage2 -= 1
        state.net += 1          # delivery to stock
        yield from review()

    def review():
        """Ask NestedSimPy for orders and launch them into the supply system."""
        # decide publishes its event ("review" by default) and that event is
        # the branch trigger: NestedSimPy launches one inner simulation per
        # (action, replication) here before this line returns.
        normal, emergency = yield from env.decide(policy, state)
        counts["normal"] += normal
        counts["emergency"] += emergency
        spend = normal * p.normal_cost + emergency * p.emergency_cost
        costs["ordering"] += spend
        env.record("cost", spend)                   # branches pay for orders
        # Pipeline counts are updated here, at order time, so that any
        # later review at the same instant already sees these orders.
        accrue()
        state.stage2 += emergency
        state.stage1 += normal
        for _ in range(emergency):
            env.process(produced_unit(emergency=True))
        for _ in range(normal):
            env.process(produced_unit(emergency=False))

    def demand_process():
        while True:
            yield env.nested_timeout(
                {"distribution": "exponential", "rate": p.demand_rate})
            accrue()
            state.net -= 1      # unmet demand is backlogged (net < 0)
            yield from review()

    def warmup_reset():
        """Discard costs incurred during the warm-up transient."""
        yield env.timeout(p.warmup)
        accrue()
        for key in costs:
            costs[key] = 0.0
        for key in counts:
            counts[key] = 0

    env.process(demand_process())
    if p.warmup > 0:        # a zero warm-up discards nothing
        env.process(warmup_reset())
    env.process(review())               # initial decision at time 0

    # No trigger configuration here: with actions declared and none given,
    # NestedSimPy branches on the event decide publishes.  Set it explicitly
    # -- set_triggering_conditions({"on": "event", "name": ...}) -- for a
    # custom event name, or to branch on something besides the decisions.
    env.set_outer_stopping_condition(timeout=p.horizon)     # end of the run
    env.set_inner_stopping_condition(relative_time=float(inner_horizon))
    env.set_inner_repetitions(inner_reps)                   # branches/action
    env.set_rng("CRN")
    env.set_outer_seed(seed)
    env.set_inner_actions(ACTIONS, metric="cost",
                          outer_run_mode="rollout")
    env.set_output_options(out_dir=out_dir, gzip_trace=False)
    env.nested_run()
    accrue()                    # charge costs for the final interval

    return {"costs": costs, "counts": counts,
            "total_cost": sum(costs.values()), "env": env}


# ---------------------------------------------------------------------------
# Main: one short transient-start demo run
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # The paper's instance with heavier demand: lambda 6.5 instead of 6,
    # so stage 2 (rate 7, which every unit passes through) runs at 93%
    # utilization and congestion bites.  Demand of 7 or more would make
    # it unstable.
    params = Params(demand_rate=6.5, horizon=25.0, warmup=0.0)
    policy = DualIndexPolicy(s1=30, s2=12)      # the paper's best DI baseline
    r = simulate(params, policy, seed=1, out_dir=NESTED_OUTPUT_FOLDER)
    print(f"rollout over {len(ACTIONS) + 1} candidates on {policy!r}: "
          f"total cost {r['total_cost']:.1f} over {params.horizon:.0f} "
          f"({r['total_cost'] / params.horizon:.2f} per unit time)")
    print(f"  ordered {r['counts']['normal']} normal, "
          f"{r['counts']['emergency']} emergency")


In [ ]:
# Run as a subprocess so the outer output is clean (inner branches run in separate processes).
!python dual_sourcing_colab.py

## 3. Inspect the run

The lookahead CSVs are read back with pandas: the executed picks and the per-action score table.

In [ ]:
import glob, os
import pandas as pd

run = os.path.dirname(glob.glob("simpy_examples/dual_sourcing/**/rollout", recursive=True)[0])

# Two of the four rollout CSVs: the executed picks, and every
# candidate's score per decision.
outer_decisions = pd.read_csv(f"{run}/rollout/outer_decisions.csv")
aggregated = pd.read_csv(f"{run}/rollout/inner_trajectories_aggregated.csv")
print(outer_decisions.to_string(index=False))  # base_policy pick = no override
print()
print(aggregated.head(8).to_string(index=False))  # base_policy row = the baseline decision, scored as one more candidate
